In [ ]:
# ==============================================================================
# 1. INSTALLS & IMPORTS
# ==============================================================================
import os
import shutil
import subprocess
from pathlib import Path
from google.colab import drive
from huggingface_hub import hf_hub_download

In [ ]:
# ==============================================================================
# 2. CONFIGURATION
#    Every setting, folder and file name used by this notebook is defined here.
# ==============================================================================

# ------------------------------------------------------------------------------
# 2a. Google Drive mount
# ------------------------------------------------------------------------------
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_MOUNT_TIMEOUT_MS = 120000

# ------------------------------------------------------------------------------
# 2b. Folders (Drive = cold storage shared with the other notebooks; Local = fast Colab disk)
# ------------------------------------------------------------------------------
DIRECTORY_NAME = "PhD-research-assistant"
DRIVE = Path(DRIVE_MOUNT_POINT) / "MyDrive"

DRIVE_DIR = DRIVE / DIRECTORY_NAME

DRIVE_WORKSPACE = DRIVE_DIR / "workspace"
DRIVE_PERSONALITY = DRIVE_DIR / "personality"
DRIVE_MODELS = DRIVE_DIR / "models"

DRIVE_MEMORY = DRIVE_PERSONALITY / "memories"
DRIVE_LOGS = DRIVE_PERSONALITY / "logs"
DRIVE_QUERIES = DRIVE_PERSONALITY / "queries"

DRIVE_PENDING = DRIVE_QUERIES / "pending"
DRIVE_CLAIMED = DRIVE_QUERIES / "claimed"
DRIVE_DONE = DRIVE_QUERIES / "done"

DRIVE_UNPROCESSED = DRIVE_WORKSPACE / "unprocessed"
DRIVE_PROCESSED = DRIVE_WORKSPACE / "processed"
DRIVE_TEMP = DRIVE_WORKSPACE / "temp-chunks"

LOCAL_OLLAMA = Path("/content/phdresearch-ollama-models")
LOCAL_MODELS = Path("/content/phdresearch-models")
LOCAL_WORKSPACE = Path("/content/phdresearch-workspace")
LOCAL_PERSONALITY = Path("/content/phdresearch-personality")

LOCAL_MEMORY = LOCAL_PERSONALITY / "memories"
LOCAL_LOGS = LOCAL_PERSONALITY / "logs"
LOCAL_QUERIES = LOCAL_PERSONALITY / "queries"

LOCAL_TEMP = LOCAL_WORKSPACE / "temp-chunks"
LOCAL_PROCESSED = LOCAL_WORKSPACE / "processed"
LOCAL_UNPROCESSED = LOCAL_WORKSPACE / "unprocessed"

directories = [
    LOCAL_TEMP, LOCAL_OLLAMA, LOCAL_WORKSPACE, LOCAL_PERSONALITY,
    LOCAL_MEMORY, LOCAL_LOGS, LOCAL_QUERIES, DRIVE_DIR, DRIVE_MEMORY, DRIVE_LOGS, DRIVE_QUERIES,
    DRIVE_PENDING, DRIVE_CLAIMED, DRIVE_DONE,
    LOCAL_PROCESSED, DRIVE_UNPROCESSED, DRIVE_PROCESSED, DRIVE_TEMP, DRIVE_MODELS
]

# ------------------------------------------------------------------------------
# 2c. Steering files (their contents are defined in section 3, FILES)
# ------------------------------------------------------------------------------
MEMORY_FILE = "MEMORY.md"
INSTRUCTIONS_FILE = "INSTRUCTIONS.md"
NUDGE_FILE = "NUDGE.md"

# ------------------------------------------------------------------------------
# 2d. Models: name used inside the pipeline -> where to download it from
# ------------------------------------------------------------------------------
REQUIRED_MODELS =
{
    "local-analyse.gguf": {
        "repo_id": "bartowski/Mistral-Small-24B-Instruct-2501-GGUF",
        "filename": "Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf"
    },
    "local-embed.gguf": {
        "repo_id": "Casual-Autopsy/snowflake-arctic-embed-l-v2.0-gguf",
        "filename": "snowflake-arctic-embed-l-v2.0-q4_k_m.gguf"
    }
}

# ------------------------------------------------------------------------------
# 2e. Mount Drive and create every folder above
#     (Drive has to be mounted before the Drive folders can be created.)
# ------------------------------------------------------------------------------
print("1. Mounting Google Drive...")
drive.mount(DRIVE_MOUNT_POINT, force_remount=True, timeout_ms=DRIVE_MOUNT_TIMEOUT_MS)

print("2. Creating local and remote directory structures...")
for p in directories: p.mkdir(parents=True, exist_ok=True)

1. Mounting Google Drive...
Mounted at /content/drive
2. Creating local and remote directory structures...


In [ ]:
# ==============================================================================
# 3. FILES
#    The text of the three steering files. Section 4 writes them to disk; the cloud agent then
#    reads them at run time (MEMORY.md and NUDGE.md also steer the query generator).
# ==============================================================================
MEMORY = """# Research Profile Context\n\n
    ## Academic Domain\n
    * **Broader Interests:** Public participation in science and research.  Equality of every individuals' right to self-determination.\n\n
    ## Synthesis Rules\n
    * **Zero-Pollution Directive:** Do not incorporate personal, demographic, or unrelated professional background details into your analysis. Keep all outputs strictly confined to academic and methodological realities.\n
    * **Text-First Principle:** The active chunk of text is the ultimate authority. Summarize only what is there. Do not inject the overarching research focus into a summary if the paper does not naturally address it."""

INSTRUCTIONS = """# Task Instructions: Evolving Literature Synthesis\n\n
    **Role:** You are an autonomous research assistant.\n\n
    **Objective:**\nYour core directive is to process semantic chunks of academic literature and continuously synthesize an evolving literature review.
    This process spans multiple ephemeral sessions and runtimes. Your output must serve as a persistent, evolving 'trail' that steers the trajectory of ongoing research.\n\n
    **Execution Steps:**
    \n1. **Analyze the Chunk:** Read the provided target semantic chunk. Evaluate its relevance to the overarching research scope.
    \n2. **Synthesize:** Contextualize the chunk against your recent memories. How does this new information shift, support, or contradict the current methodological trajectory?
    \n3. **Distill:** Formulate exactly one highly condensed, rigorously academic sentence that captures the core utility of this chunk for the final literature review.\n\n---\n
    **CRITICAL DIRECTIVES:**\n1. You must conclude your synthesis with the exact heading `### Auto-Memory:` followed by exactly ONE sentence stating the specific methodology, insight, or parameter to be added to the permanent literature review.
    \n2. DO NOT USE TOOLS. You are strictly forbidden from invoking any external tools, APIs, or functions. Provide only a raw text analysis."""

NUDGE = """archaeogaming"""

# Which prompt is written to which file (file names come from the config section)
core_files = {
    MEMORY_FILE: MEMORY,
    INSTRUCTIONS_FILE: INSTRUCTIONS,
    NUDGE_FILE: NUDGE,
}

In [ ]:
# ==============================================================================
# 4. CORE CONTEXT INJECTION
# ==============================================================================
print("3. Injecting core instructions and active memory states...")

for filename, content in core_files.items():
    local_file = LOCAL_PERSONALITY / filename
    drive_file = DRIVE_PERSONALITY / filename

    # Write to local NVMe for fast access
    local_file.write_text(content.strip() + "\n", encoding="utf-8")

    # Push to Drive for cold storage (only if missing, to avoid overwriting evolved states)
    if not drive_file.exists():
        shutil.copy2(local_file, drive_file)

3. Injecting core instructions and active memory states...


In [ ]:
# ==============================================================================
# 5. MODEL VERIFICATION & DOWNLOAD (LOCAL-FIRST PROTOCOL)
# ==============================================================================
print("4. Verifying offline models in cold storage...")


for target_name, details in REQUIRED_MODELS.items():
    drive_path = DRIVE_MODELS / target_name
    local_path = LOCAL_MODELS / target_name

    if drive_path.exists():
        print(f"[✓] {target_name} is already safely stored on Google Drive.")
    else:
        print(f"[-] Missing {target_name} on Drive. Downloading to local SSD first...")
        try:
            # 1. Download to local Colab NVMe to bypass Drive FUSE cache drops
            downloaded_path = hf_hub_download(
                repo_id=details["repo_id"],
                filename=details["filename"],
                local_dir=str(LOCAL_MODELS)
            )

            # 2. Rename to our standard pipeline convention locally
            Path(downloaded_path).rename(local_path)

            # 3. Transfer the completed, verified file to Google Drive
            print(f"[+] Download complete. Transferring to Google Drive (this may take a few minutes)...")
            shutil.copy2(local_path, drive_path)

            # 4. Delete the local copy to free up Colab disk space (Ollama will ingest from Drive)
            local_path.unlink()

            print(f"[✓] Successfully backed up {target_name} to Drive.")
        except Exception as e:
            print(f"[!] Failed to download or transfer {target_name}: {e}")

print("\n[✓] Environment setup complete. The agent pipeline is ready for launch.")

4. Verifying offline models in cold storage...
[-] Missing mistral-small-24b.gguf on Drive. Downloading to local SSD first...


Mistral-Small-24B-Instruct-2501-Q4_K_M.g(…): reconstructing file:   0%|          |  0.00B / 14.3GB            

Mistral-Small-24B-Instruct-2501-Q4_K_M.g(…): downloading bytes:           |  0.00B            

[+] Download complete. Transferring to Google Drive (this may take a few minutes)...
[✓] Successfully backed up mistral-small-24b.gguf to Drive.
[-] Missing local-embed.gguf on Drive. Downloading to local SSD first...


snowflake-arctic-embed-l-v2.0-q4_k_m.ggu(…): reconstructing file:   0%|          |  0.00B /  438MB            

snowflake-arctic-embed-l-v2.0-q4_k_m.ggu(…): downloading bytes:           |  0.00B            

[+] Download complete. Transferring to Google Drive (this may take a few minutes)...
[✓] Successfully backed up local-embed.gguf to Drive.

[✓] Environment setup complete. The agent pipeline is ready for launch.
